# The EYE/N3 certificate checker, in a notebook

`eye/check.py` is a Python module. This notebook **imports it and calls its
functions**; there is no shell command and no CLI in the loop. A verdict is a
function call:

```
check_certificate(asserted, steps, ...)  ->  (exit_code, result_dict)
```

The exit codes are the ones the command line uses (`0` accept, `1` rejected,
`2` unreadable, and `3` when the proven anchor disagrees), so a pipeline and a
notebook read the same contract — but the notebook does not spawn a process to
get them.

**What an `accept` means.** `check.py` is the untrusted driver; the trusted part
is `checker.n3`, run by the EYE reasoner. EYE carries **no machine-checked
theorem** (`OOCert.certificate_sound` is about the Lean function and is not
reproved here). An accept means *the rules licensed every step*, not *it is
entailed*. With the Rocq anchor set — and only for `oo-horn` — the accept gains
the theorem.

**No npm.** The reasoner is the native EYE image (`lib/eye.pvm`, built by
`build_eye.sh` with SWI-Prolog), which is the same EYE the npm package embeds.
Cell 0 prints which reasoner was found.

## 0 · Import the checker

In [1]:
import importlib.util, json
from pathlib import Path

# the notebook lives in eye/examples/; the checker is one level up
EYE = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
CHECK = EYE / "check.py"
O = EYE.parent
FIX = O / "tests" / "fixtures"
ROCQ = O / "rocq" / "build" / "oo-horn-rocq"
assert CHECK.exists(), f"no check.py at {CHECK}"

spec = importlib.util.spec_from_file_location("eyecheck", CHECK)
eye = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eye)

print("imported :", eye.__name__, "from", CHECK)
print("reasoner :", eye.reasoner_cmd())      # native SWI-Prolog image, not npx
print("fixtures :", FIX)
print("anchor   :", ROCQ, "(built)" if ROCQ.exists() else "(not built)")

imported : eyecheck from /workspaces/open-ontologies/eye/check.py
reasoner : ['/usr/bin/swipl', '-x', '/workspaces/open-ontologies/eye/lib/eye.pvm', '--']
fixtures : /workspaces/open-ontologies/tests/fixtures
anchor   : /workspaces/open-ontologies/rocq/build/oo-horn-rocq (built)


## 0b · Build the Rocq checker if it is not there yet

The anchor is a **build artifact** (`rocq/build/oo-horn-rocq`), gitignored, and
so is often absent on a fresh checkout. The cell below builds it **only when
needed** — missing, un-runnable (a host-arch binary in a shared tree gives
`Exec format error`), or failing its own smoke test — and does nothing when a
working one is already present. This is a system build, which is the one job a
shell cell is for; every *check* in this notebook stays an imported-function
call.

In [2]:
%%bash
# Build the Rocq-extracted checker only if it is missing or unusable.
# The notebook runs from eye/examples/, so the repo root is ../..
set -u
O="$(cd "$(pwd)/../.." && pwd)"
BIN="$O/rocq/build/oo-horn-rocq"

ok=0
if [ ! -x "$BIN" ]; then
  echo "anchor: not built"
elif ! "$BIN" 2>&1 | grep -q 'usage: oo-horn-rocq'; then
  echo "anchor: present but not runnable (wrong architecture or corrupt)"
else
  echo "anchor: present and runnable"
  ok=1
fi

if [ "$ok" = 0 ]; then
  if command -v rocq >/dev/null 2>&1; then
    echo "building: cd rocq && bash build.sh"
    ( cd "$O/rocq" && bash build.sh ) && echo "anchor: built" || echo "anchor: BUILD FAILED"
  else
    echo "anchor: rocq not on PATH; cannot build here."
    echo "  inside the devcontainer this is installed; otherwise:"
    echo "  apt-get install -y coq libcoq-stdlib ocaml ocaml-findlib"
  fi
fi


anchor: present and runnable


## 1 · The verdict is a function

`check_certificate(asserted, steps, fmt=..., cert_path=..., asserted_path=...,
anchor_binary=..., anchor_rules=...)` returns `(exit_code, result_dict)`. It is
the same code the CLI calls; the CLI is now a thin adapter over it (`main()`
reads two files and prints the dict). Two helpers wrap it for readability — they
decide nothing.

In [3]:
def check_files(asserted_path, cert_path, **kw):
    """Read two files, call check_certificate, return (code, dict)."""
    asserted = eye.parse_asserted(Path(asserted_path).read_text())
    fmt, steps, refute_line = eye.load_derivations(Path(cert_path))
    return eye.check_certificate(
        asserted, steps, fmt=fmt, refute_line=refute_line,
        cert_path=str(cert_path), asserted_path=str(asserted_path), **kw)

def check_text(asserted_text, deriv_text, fmt="oo-cert"):
    """Same contract, for fixtures built as text."""
    asserted = eye.parse_asserted(asserted_text)
    steps = eye.parse_steps(deriv_text) if fmt == "oo-cert" else eye.parse_horn_steps(deriv_text)
    return eye.check_certificate(asserted, steps, fmt=fmt)

def show(label, asserted_path, cert_path, **kw):
    code, out = check_files(asserted_path, cert_path, **kw)
    print(f"{label:<46} exit={code}")
    print("   ", json.dumps(out)[:200])
    return code, out

### 1a · An honest certificate is accepted

In [4]:
show("honest rdfs9 certificate", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")

honest rdfs9 certificate                       exit=0
    {"ok": true, "asserted": 4, "derivations": 1}


(0, {'ok': True, 'asserted': 4, 'derivations': 1})

### 1b · A forged one is refused

Same certificate, change **only the conclusion object** (`Carnivore` ->
`Herbivore`), both premises exactly as they were.

In [5]:
import re
src = (FIX / "refute/derivations.tsv").read_text()
forged = re.sub("Carnivore", "Herbivore", src, count=1)   # first occurrence only
forged_path = EYE / "examples" / "forged_derivations.tsv"
forged_path.write_text(forged)
print("before:", src.splitlines()[0].split("\t")[:4])
print("after :", forged.splitlines()[0].split("\t")[:4]); print()
show("forged certificate", FIX / "refute/asserted.tsv", forged_path)

before: ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Carnivore>']
after : ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Herbivore>']

forged certificate                             exit=1
    {"ok": false, "rejected_step": 0, "rule": "rdfs9", "conclusion": "<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>", "premises": ["<http://ex.org/leo> <h


(1,
 {'ok': False,
  'rejected_step': 0,
  'rule': 'rdfs9',
  'conclusion': '<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>',
  'premises': ['<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Lion>',
   '<http://ex.org/Lion> <http://www.w3.org/2000/01/rdf-schema#subClassOf> <http://ex.org/Carnivore>']})

## 2 · Two certificate kinds (auto-detected)

- **`oo-cert`** — `derivations.tsv`: the OWL-RL rule set (29 arms).
- **`oo-horn`** — `horn.tsv`: the Horn rule set (the Rocq-proven one).

`load_derivations` picks the parser from the file's own shape.

In [6]:
show("oo-horn (Horn rule set)", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")

oo-horn (Horn rule set)                        exit=0
    {"ok": true, "asserted": 2, "derivations": 1}


(0, {'ok': True, 'asserted': 2, 'derivations': 1})

## 3 · The contract cases, in Python

This is `run_tests.sh`'s job done in the notebook: the same positive and
negative cases, the negative fixtures built inline as text, every check run
through the imported module. No shell script, no subprocess.

| case | what it proves |
|---|---|
| positive `oo-cert` | a real certificate is accepted |
| positive `oo-horn` | a Horn certificate is accepted |
| forged | one changed conclusion in a real certificate is refused |
| (a) premise asserted nowhere | an unbacked premise is refused |
| (b) premises in wrong order | order is part of the contract |
| (c) `scm-avf2` natural direction | the one reversed rule is enforced the right way round |
| (d) `cls-int1` missing a member type | every list member must be typed |

In [7]:
from textwrap import dedent

# The RDF vocabularies, spelled once and readably.
RDF  = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
RDFS = "http://www.w3.org/2000/01/rdf-schema#"
OWL  = "http://www.w3.org/2002/07/owl#"

TYPE         = f"<{RDF}type>"
FIRST        = f"<{RDF}first>"
REST         = f"<{RDF}rest>"
NIL          = f"<{RDF}nil>"
SUB_CLASS    = f"<{RDFS}subClassOf>"
SUB_PROPERTY = f"<{RDFS}subPropertyOf>"
ALL_VALUES   = f"<{OWL}allValuesFrom>"
ON_PROPERTY  = f"<{OWL}onProperty>"
INTERSECTION = f"<{OWL}intersectionOf>"


def graph(block):
    """An asserted graph as a block of triples, one per line, whitespace-
    separated: the file the checker reads is those triples, tab-separated."""
    rows = dedent(block).strip().splitlines()
    return "\n".join("\t".join(row.split()) for row in rows)


def step(rule, conclusion, *premises):
    """One derivation line: the rule, the conclusion triple, then each premise
    triple, in the order the rule documents. Written as triples so a long step
    stays readable instead of becoming one crammed string."""
    fields = [rule, *conclusion]
    for premise in premises:
        fields.extend(premise)
    return "\t".join(fields)


# (a) a premise asserted nowhere and concluded nowhere
A_a = graph("""
    <http://ex.org/B> {SUB_CLASS} <http://ex.org/C>
""")
D_a = step("rdfs9",
           ("<http://ex.org/a>", TYPE, "<http://ex.org/C>"),
           ("<http://ex.org/a>", TYPE, "<http://ex.org/B>"),
           ("<http://ex.org/B>", SUB_CLASS, "<http://ex.org/C>"))

# (b) rdfs9 with the premises in the wrong order (both asserted)
A_b = graph("""
    <http://ex.org/a> {TYPE} <http://ex.org/A>
    <http://ex.org/A> {SUB_CLASS} <http://ex.org/B>
""")
D_b = step("rdfs9",
           ("<http://ex.org/a>", TYPE, "<http://ex.org/B>"),
           ("<http://ex.org/A>", SUB_CLASS, "<http://ex.org/B>"),
           ("<http://ex.org/a>", TYPE, "<http://ex.org/A>"))

# (c) scm-avf2 with the conclusion in the natural (wrong) direction c1 sc c2
A_c = graph("""
    <http://ex.org/C1> {ALL_VALUES} <http://ex.org/Y>
    <http://ex.org/C1> {ON_PROPERTY} <http://ex.org/p1>
    <http://ex.org/C2> {ALL_VALUES} <http://ex.org/Y>
    <http://ex.org/C2> {ON_PROPERTY} <http://ex.org/p2>
    <http://ex.org/p1> {SUB_PROPERTY} <http://ex.org/p2>
""")
D_c = step("scm-avf2",
           ("<http://ex.org/C1>", SUB_CLASS, "<http://ex.org/C2>"),
           ("<http://ex.org/C1>", ALL_VALUES, "<http://ex.org/Y>"),
           ("<http://ex.org/C1>", ON_PROPERTY, "<http://ex.org/p1>"),
           ("<http://ex.org/C2>", ALL_VALUES, "<http://ex.org/Y>"),
           ("<http://ex.org/C2>", ON_PROPERTY, "<http://ex.org/p2>"),
           ("<http://ex.org/p1>", SUB_PROPERTY, "<http://ex.org/p2>"))

# (d) cls-int1 with one member's type missing
A_d = graph("""
    <http://ex.org/c> {INTERSECTION} <http://ex.org/l>
    <http://ex.org/l> {FIRST} <http://ex.org/A>
    <http://ex.org/l> {REST} <http://ex.org/l2>
    <http://ex.org/l2> {FIRST} <http://ex.org/B>
    <http://ex.org/l2> {REST} {NIL}
    <http://ex.org/x> {TYPE} <http://ex.org/A>
""")
D_d = step("cls-int1",
           ("<http://ex.org/x>", TYPE, "<http://ex.org/c>"),
           ("<http://ex.org/c>", INTERSECTION, "<http://ex.org/l>"),
           ("<http://ex.org/l>", FIRST, "<http://ex.org/A>"),
           ("<http://ex.org/l>", REST, "<http://ex.org/l2>"),
           ("<http://ex.org/l2>", FIRST, "<http://ex.org/B>"),
           ("<http://ex.org/l2>", REST, NIL),
           ("<http://ex.org/x>", TYPE, "<http://ex.org/A>"))

ran = fails = 0
def expect(want_code, label, code, out):
    global ran, fails
    ran += 1
    good = (code == want_code)
    verdict = "accept" if code == 0 else f"exit {code}"
    print(f"{'PASS' if good else 'FAIL'} [{label}] {verdict}")
    if not good:
        fails += 1
        print("      ", json.dumps(out)[:180])

# positives and the forged real certificate (through check_certificate)
for label, a, c, want in [
    ("pos: oo-cert",           FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv",   0),
    ("pos: oo-horn",           FIX / "horn/asserted.tsv",   FIX / "horn/good.tsv",             0),
    ("neg: forged conclusion", FIX / "refute/asserted.tsv", forged_path,                       1),
    ("neg: oo-horn forged",    FIX / "horn/asserted.tsv",   FIX / "horn/supplier/forged.tsv",  1),
]:
    code, out = check_files(a, c)
    expect(want, label, code, out)

# contract cases (built as text)
for label, a, d, want in [
    ("neg(a): unasserted/underived premise", A_a, D_a, 1),
    ("neg(b): premises in wrong order",       A_b, D_b, 1),
    ("neg(c): scm-avf2 natural direction",    A_c, D_c, 1),
    ("neg(d): cls-int1 missing member type",  A_d, D_d, 1),
]:
    code, out = check_text(a, d)
    expect(want, label, code, out)

print(f"\nran {ran}, failed {fails}")
assert fails == 0, "contract cases failed"


PASS [pos: oo-cert] accept
PASS [pos: oo-horn] accept
PASS [neg: forged conclusion] exit 1
PASS [neg: oo-horn forged] exit 1
PASS [neg(a): unasserted/underived premise] exit 1
PASS [neg(b): premises in wrong order] exit 1
PASS [neg(c): scm-avf2 natural direction] exit 1
PASS [neg(d): cls-int1 missing member type] exit 1

ran 8, failed 0


### 3a · `oo-refute/1` is a different sentence

A refutation file starts with the tag `oo-refute/1`. It is **not** an `oo-cert`
certificate, so it earns a different verdict: `check_certificate` reports it and
does not check it as one (code `2`).

In [8]:
fmt, steps, refute_line = eye.load_derivations(FIX / "refute/good.tsv")
print("format    :", fmt)
print("step count:", len(steps))
print("refute    :", refute_line)
asserted = eye.parse_asserted((FIX / "refute/asserted.tsv").read_text())
code, out = eye.check_certificate(asserted, steps, fmt=fmt, refute_line=refute_line)
print("verdict   : exit", code, "--", out["error"])
assert fmt == "oo-refute" and code == 2

format    : oo-refute
step count: 1
refute    : refute	cax-dw	<http://ex.org/Herbivore>	<http://www.w3.org/2002/07/owl#disjointWith>	<http://ex.org/Carnivore>	<http://ex.org/leo>	<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>	<http://ex.org/Herbivore>	<http://ex.org/leo>	<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>	<http://ex.org/Carnivore>
verdict   : exit 2 -- this is an oo-refute/1 refutation certificate; oo-cert does not check it (different verdict)


## 4 · The optional proven anchor (Rocq)

Pass `anchor_binary` and `anchor_rules` to `check_certificate` (the CLI sets them
from `OO_HORN_ROCQ`/`OO_RULES`). It runs the **Rocq-extracted, machine-checked**
`oo-horn` checker over the same inputs:

- agree -> the accept carries the theorem name;
- disagree on an `oo-horn` certificate -> code **3**, not 0;
- on an `oo-cert` certificate the anchor **declines** (Horn-only).

In [9]:
ANCHOR = dict(anchor_binary=str(ROCQ), anchor_rules=str(FIX / "horn/builtin_rules.tsv"))

if ROCQ.exists():
    show("oo-horn good + anchor", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv", **ANCHOR)
    print()
    show("oo-horn bad_conclusion + anchor", FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv", **ANCHOR)
    print()
    show("oo-cert + anchor (declines)", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv", **ANCHOR)
else:
    print("The Rocq anchor is not built here. Build it with:")
    print("  (cd", O / "rocq", "&& bash build.sh)")

oo-horn good + anchor                          exit=0
    {"ok": true, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 0, "verdict": {"ok": true, "kernel": "rocq", "verdict": "entailed", "rules": 27, "asserted": 2, "derivations"

oo-horn bad_conclusion + anchor                exit=3
    {"ok": false, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 1, "verdict": {"ok": false, "kernel": "rocq", "rules": 27, "asserted": 2, "derivations": 1}}, "anchor_disagr

oo-cert + anchor (declines)                    exit=0
    {"ok": true, "asserted": 4, "derivations": 1, "anchor": {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}}


### Why `bad_conclusion` matters

The fixture has a binding `b = Z` contradicting its own conclusion object `B`.
EYE **accepts** it (the Horn bindings are dropped when the rule index is mapped
through `asHorn`); the proven checker **refuses** it. A textbook **false pass** —
surfaced only by running the two together. The code is **3**, not 0, so a pipeline
that checks for success cannot mistake it for one.

## 5 · The exit-code contract (still a function call)

Everything above was a call to `check_certificate`, which **returns** the code a
shell would read from `$?`. There is no subprocess anywhere in this notebook.
The four codes, from four calls:

In [10]:
results = []
results.append(("honest oo-cert",           check_files(FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")))
results.append(("forged oo-cert",           check_files(FIX / "refute/asserted.tsv", forged_path)))
if ROCQ.exists():
    results.append(("oo-horn good + anchor",    check_files(FIX / "horn/asserted.tsv", FIX / "horn/good.tsv", **ANCHOR)))
    results.append(("oo-horn bad + anchor",     check_files(FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv", **ANCHOR)))
    want = [0, 1, 0, 3]
else:
    want = [0, 1]
for (label, (code, out)), w in zip(results, want):
    print(f"{label:<30} return {code}  (want {w})")
    assert code == w, label
print()
print("0 accept | 1 rejected | 2 unreadable | 3 anchor disagreement")

honest oo-cert                 return 0  (want 0)
forged oo-cert                 return 1  (want 1)
oo-horn good + anchor          return 0  (want 0)
oo-horn bad + anchor           return 3  (want 3)

0 accept | 1 rejected | 2 unreadable | 3 anchor disagreement


## 6 · What this does *not* claim

- The `oo-cert` soundness theorem (`OOCert.certificate_sound`) is about the
  **Lean** function and is **not** reproved here. For `oo-cert`, an accept is
  *the rules licensed every step*.
- The Rocq anchor proves a theorem about the **Horn** rule set only
  (`entails_of_builtin_horn`); it applies to `oo-horn`, not `oo-cert`.
- Even with the anchor, the verdict is **conditional on the asserted graph**:
  neither EYE nor Rocq checks the graph is the one the engine actually reasoned
  over.

That is the honest boundary of this notebook, and of `eye/` itself.